# Module 2: Snowflake Postgres — Customer Self-Service Portal

This module extends the EPOWER demo with a **Snowflake Postgres** instance that backs the **"Mein EPOWER"** customer self-service portal — a web application where 20,000 customers manage their energy accounts online.

---

### The Business Case

Every energy retailer needs a digital customer portal. In Germany, where EPOWER operates, regulatory requirements (Marktkommunikation) and customer expectations demand self-service capabilities: meter reading submission, tariff switching, billing inquiries, and program enrollments.

**"Mein EPOWER"** is EPOWER's customer-facing web application. Behind it: a PostgreSQL database handling the transactional workload — logins, form submissions, order processing, and session management for 20,000 customers.

The challenge: **How do you get operational portal data into your analytical platform without building ETL pipelines?** The answer: Snowflake Postgres + pg_lake. The portal writes to Postgres for operations; Postgres writes to Iceberg for analytics; Snowflake reads Iceberg natively. Zero middleware.

---

### Why Postgres?

The portal is a standard web application stack: **React frontend → REST API → PostgreSQL**. This is the most common backend pattern in software — used by millions of applications worldwide.

| Portal Feature | OLTP Requirement | Why Not Snowflake? |
|---------------|-----------------|-------------------|
| **Submit meter readings** | Validation (new ≥ previous), INSERT with constraints | Sub-second response needed for UX |
| **Request tariff switch** | Atomic order with status lifecycle (PENDING → CONFIRMED → ACTIVE) | Row-level locking for concurrent status updates |
| **Open service request** | INSERT with auto-categorization | Hundreds of concurrent form submissions |
| **VPP enrollment** | Multi-step signup with rollback on failure | Transaction semantics (all-or-nothing) |
| **Login & sessions** | Concurrent auth, session UPDATEs, CSRF tokens | Millisecond reads for session validation |

---

### The Operational → Analytical Bridge

| Data | Lives in Postgres | Flows to Snowflake? | Why? |
|------|------------------|--------------------| -----|
| User sessions & auth | ✅ | ❌ | Ephemeral, no analytical value |
| Meter readings (raw) | ✅ | ❌ | Already in Snowflake via billing pipeline |
| Tariff orders (mutable) | ✅ | ❌ | Status changes frequently, needs row-locking |
| **Portal activity log** | ✅ | **✅ via pg_lake** | Append-only, denormalized, analytically rich |

The `portal_activity_log` is the bridge — every user action generates one immutable log entry. This append-only stream is the natural replication target for analytics.

---

### Architecture

```
┌───────────────────────────────────────────────────────────────────┐
│  "Mein EPOWER" Portal (Web Application)                           │
│   React Frontend → REST API → Snowflake Postgres                  │
│                                                                   │
│  ┌──────────────┐  ┌────────────────┐  ┌──────────────────────┐  │
│  │ portal_users │  │ meter_readings │  │ tariff_orders         │  │
│  │ (sessions)   │  │ (kWh data)     │  │ service_requests      │  │
│  └──────────────┘  └────────────────┘  └──────────────────────┘  │
│                                                                   │
│       Every user action → portal_activity_log (append-only)       │
│                                    │                              │
│                         pg_incremental (1 min)                    │
│                                    ▼                              │
│                         ┌────────────────────┐                    │
│                         │  Iceberg table      │                    │
│                         │  (pg_lake managed)  │                    │
│                         └─────────┬──────────┘                    │
└───────────────────────────────────┼───────────────────────────────┘
                                    │
                         Catalog Integration (auto-refresh 30s)
                                    ▼
┌───────────────────────────────────────────────────────────────────┐
│  SNOWFLAKE (Analytics + AI)                                       │
│  EPOWER_BRONZE → PORTAL_ACTIVITY_LOG (Iceberg)                    │
│  EPOWER_GOLD   → MART_PORTAL_ENGAGEMENT                          │
│               → PORTAL_SEMANTIC_VIEW → EPOWER AGENT               │
└───────────────────────────────────────────────────────────────────┘
```

---

### Snowflake Features Introduced

| Feature | What It Is | Role in This Module |
|---------|-----------|-------------------|
| **Snowflake Postgres** | Fully managed PostgreSQL. Connects via `psql`, JDBC, or any ORM. | Portal transactional backend |
| **pg_lake** | Postgres extension for native Iceberg table support | Creates Iceberg table for the activity log |
| **pg_incremental** | Automated, exactly-once incremental pipelines | Syncs activity log heap → Iceberg every minute |
| **Catalog Integration** | Snowflake reads external Iceberg catalogs | Bridge from Postgres-managed Iceberg to Snowflake |
| **Auto-Refresh** | Snowflake polls Iceberg catalog for new snapshots | Near real-time portal data in Snowflake |

---

### What You'll Learn

| Section | What we do |
|---------|------------|
| **§1** Prerequisites | Verify Module 1 is deployed |
| **§2** Postgres Instance | Provision Snowflake Postgres |
| **§3** psql Setup | Install psql, save connection |
| **§4** Portal Schema + Data | Create tables and seed data (in psql) |
| **§5** Catalog Integration | Connect Snowflake to Postgres Iceberg |
| **§6** Analytics Model | Create engagement metrics |
| **§7** Semantic View + Agent | Add portal_analyst tool |
| **§8** Verification & Demo | Validate pipeline, demo live sync |

**Runtime**: ~15 minutes | **Prerequisite**: Module 1 (`epower_hol.ipynb`) must be deployed

## 1. Prerequisites

This module requires Module 1 (`epower_hol.ipynb`) to be fully deployed — we need the customer dimension and product data.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected as: {session.get_current_user()}")

In [ ]:
%%sql
USE ROLE EPOWER_ROLE;
USE WAREHOUSE EPOWER_COMPUTE;
USE DATABASE EPOWER_DEMO;

SELECT 'CUSTOMER_DIM' AS required_object, COUNT(*) AS rows FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM
UNION ALL SELECT 'PRODUCT_DIM', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM
UNION ALL SELECT 'CUSTOMER_PRODUCTS', COUNT(*) FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_PRODUCTS
ORDER BY required_object;

## 2. Create Snowflake Postgres Instance

We provision a Snowflake Postgres instance to serve as the **portal backend** — the transactional database behind the "Mein EPOWER" web application.

&nbsp;

> **Snowflake Feature:** Snowflake Postgres is fully managed PostgreSQL. The portal's web application connects to it like any standard Postgres database — via `psql`, JDBC, or any ORM. No infrastructure to manage.

In [ ]:
%%sql
CREATE POSTGRES INSTANCE IF NOT EXISTS MY_EPOWER_PORTAL
    COMPUTE_FAMILY = 'STANDARD_M'
    STORAGE_SIZE_GB = 10
    AUTHENTICATION_AUTHORITY = POSTGRES
    AUTO_SUSPEND_SECS = 600
    COMMENT = 'Mein EPOWER — customer self-service portal backend (20K customers)';

-- IMPORTANT: Save the credentials from the output! They cannot be retrieved later.

In [ ]:
%%sql
CREATE NETWORK RULE IF NOT EXISTS EPOWER_PG_INGRESS
    TYPE = IPV4
    VALUE_LIST = ('0.0.0.0/0')
    MODE = POSTGRES_INGRESS;

CREATE NETWORK POLICY IF NOT EXISTS EPOWER_PG_POLICY
    ALLOWED_NETWORK_RULE_LIST = ('EPOWER_PG_INGRESS');

ALTER POSTGRES INSTANCE MY_EPOWER_PORTAL
    SET NETWORK_POLICY = 'EPOWER_PG_POLICY';

DESCRIBE POSTGRES INSTANCE MY_EPOWER_PORTAL;

## 3. Connect to Postgres via psql

The Postgres schema, data loading, and pg_lake setup are done in a **standard PostgreSQL client**. This mirrors real-world usage: application developers work in their PG tools, data engineers work in Snowflake.

---

### Install psql on macOS

```bash
brew install libpq
brew link --force libpq
```

Verify: `psql --version` should show PostgreSQL 16+.

**Alternative GUI clients:** [DBeaver Community](https://dbeaver.io/) (free, cross-platform), [pgAdmin 4](https://www.pgadmin.org/).

---

### Save Connection (recommended)

To avoid typing the long connection string every time, save it in PostgreSQL's standard service file.

**1. Create/edit `~/.pg_service.conf`:**

```ini
[my_epower_portal]
host=<HOST from DESCRIBE output above>
port=5432
dbname=postgres
user=snowflake_admin
sslmode=require
```

**2. Save password in `~/.pgpass`:**

```
<HOST>:5432:postgres:snowflake_admin:<PASSWORD from CREATE output>
```

Then set permissions: `chmod 600 ~/.pgpass`

**3. Connect:**

```bash
psql service=my_epower_portal
```

---

### Quick Connect (one-liner)

If you prefer not to save the connection:

```bash
psql "host=<HOST> port=5432 dbname=postgres user=snowflake_admin sslmode=require"
```

You'll be prompted for the password.

## 4. Portal Schema + Data (Postgres Client)

This section is executed in your **Postgres client** (psql, DBeaver, pgAdmin), not in this notebook.

### Step 1: Create the schema + pg_lake setup

Run the static setup file:

```bash
psql service=my_epower_portal -f portal_postgres_setup.sql
```

This file (`hol/portal_postgres_setup.sql`) creates:
- 5 tables: `portal_users`, `meter_readings`, `tariff_orders`, `service_requests`, `portal_activity_log`
- Indexes for common query patterns
- pg_lake + pg_cron + pg_incremental extensions
- `portal_activity_log_iceberg` (Iceberg mirror table)
- Automated pg_incremental pipeline (syncs every 1 minute)

> **Note:** The pg_incremental pipeline starts syncing from `min(event_time)` in `portal_activity_log`. Run the seed data (Step 2) before this file, OR run this file first — the pipeline will pick up data once it exists.

### Step 2: Generate and load seed data

Run the **next cell** in this notebook to generate `portal_seed_data.sql` from your Snowflake CUSTOMER_DIM and PRODUCT_DIM tables. Then load it into Postgres:

```bash
psql service=my_epower_portal -f portal_seed_data.sql
```

### Step 3: Verify in Postgres

```sql
SELECT 'portal_users' AS tbl, count(*) AS rows FROM portal_users
UNION ALL SELECT 'meter_readings', count(*) FROM meter_readings
UNION ALL SELECT 'tariff_orders', count(*) FROM tariff_orders
UNION ALL SELECT 'service_requests', count(*) FROM service_requests
UNION ALL SELECT 'portal_activity_log', count(*) FROM portal_activity_log
ORDER BY tbl;
```

### Step 4: Verify Iceberg sync

After ~2 minutes (pg_incremental backfill), check that the Iceberg table has data:

```sql
SELECT
    (SELECT count(*) FROM portal_activity_log) AS heap_rows,
    (SELECT count(*) FROM portal_activity_log_iceberg) AS iceberg_rows;
```

Both counts should match. If `iceberg_rows` is 0, wait another minute.

In [ ]:
import random, os

customers_df = session.sql("""
    SELECT customer_key, customer_name, city, state AS region, customer_type
    FROM EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_DIM
    ORDER BY customer_key
""").collect()

products_df = session.sql("""
    SELECT product_key, product_name, category_name
    FROM EPOWER_DEMO.EPOWER_GOLD.PRODUCT_DIM
""").collect()

print(f"Loaded {len(customers_df)} customers and {len(products_df)} products")

electricity_products = [p['PRODUCT_NAME'] for p in products_df if p['CATEGORY_NAME'] in ('Strom',)]
request_types = ['SUPPORT', 'COMPLAINT', 'VPP_ENROLLMENT', 'VPP_OPTOUT', 'BILLING_INQUIRY', 'MOVE']
request_subjects = {
    'SUPPORT': ['Zählerstand korrigieren', 'Login-Probleme', 'App funktioniert nicht', 'Abschlag ändern'],
    'COMPLAINT': ['Rechnung unklar', 'Zu hoher Abschlag', 'Mahnung trotz Zahlung', 'Kein Rückruf erhalten'],
    'VPP_ENROLLMENT': ['ePulse VPP Anmeldung', 'VPP Programm beitreten'],
    'VPP_OPTOUT': ['VPP Programm kündigen', 'ePulse deaktivieren'],
    'BILLING_INQUIRY': ['Rechnung nachfragen', 'Gutschrift fehlt', 'Zahlungsnachweis'],
    'MOVE': ['Umzug melden', 'Adresse ändern', 'Vertrag mitnehmen']
}

power_users = set(random.sample(range(len(customers_df)), int(len(customers_df) * 0.10)))
regular_users = set(random.sample([i for i in range(len(customers_df)) if i not in power_users], int(len(customers_df) * 0.40)))
occasional_users = set(random.sample([i for i in range(len(customers_df)) if i not in power_users and i not in regular_users], int(len(customers_df) * 0.30)))

sql_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'portal_seed_data.sql')

with open(sql_path, 'w') as f:
    f.write('-- Generated portal seed data (from EPOWER CUSTOMER_DIM + PRODUCT_DIM)\n')
    f.write('-- Run: psql service=my_epower_portal -f portal_seed_data.sql\n\n')
    f.write('BEGIN;\n\n')

    f.write('-- Portal users\n')
    user_values = []
    for c in customers_df:
        ck = c['CUSTOMER_KEY']
        name = c['CUSTOMER_NAME'].replace("'", "''")
        email = f"kunde{ck}@epower-portal.de"
        days_ago = random.randint(60, 365)
        user_values.append(f"({ck}, '{email}', '{name}', now() - interval '{days_ago} days', 'de', TRUE)")
    for i in range(0, len(user_values), 500):
        batch = user_values[i:i+500]
        f.write(f"INSERT INTO portal_users (customer_key, email, display_name, registered_at, preferred_language, notifications_enabled) VALUES\n")
        f.write(',\n'.join(batch))
        f.write('\nON CONFLICT (customer_key) DO NOTHING;\n\n')

    activity_values = []
    reading_values = []
    order_values = []
    request_values = []

    print("Generating 60 days of portal activity...")
    for day_offset in range(60, 0, -1):
        for idx, c in enumerate(customers_df):
            if idx in power_users:
                acts_today = random.random() < 0.15
            elif idx in regular_users:
                acts_today = random.random() < 0.05
            elif idx in occasional_users:
                acts_today = random.random() < 0.015
            else:
                acts_today = random.random() < 0.003

            if not acts_today:
                continue

            ck = c['CUSTOMER_KEY']
            city = c['CITY'].replace("'", "''") if c['CITY'] else 'Unknown'
            region = c['REGION'].replace("'", "''") if c['REGION'] else 'Unknown'
            ctype = c['CUSTOMER_TYPE'].replace("'", "''") if c['CUSTOMER_TYPE'] else 'Unknown'
            ts = f"now() - interval '{day_offset} days' + interval '{random.randint(6,22)} hours {random.randint(0,59)} minutes'"

            action = random.choices(['LOGIN', 'METER_READING', 'TARIFF_CHANGE', 'SERVICE_REQUEST'], weights=[50, 25, 10, 15])[0]

            if action == 'LOGIN':
                detail = '{"action": "login"}'
            elif action == 'METER_READING':
                meter_type = random.choice(['ELECTRICITY', 'GAS'])
                kwh = random.randint(2000, 35000)
                detail = f'{{"meter_type": "{meter_type}", "reading_kwh": {kwh}}}'
                reading_values.append(f"({ck}, CURRENT_DATE - {day_offset}, '{meter_type}', {kwh}, {ts}, 'PORTAL')")
            elif action == 'TARIFF_CHANGE':
                new_product = random.choice(electricity_products) if electricity_products else 'Ökostrom 100%'
                detail = f'{{"requested_product": "{new_product}"}}'
                status = random.choice(['PENDING', 'CONFIRMED', 'ACTIVE', 'CANCELLED'])
                order_values.append(f"({ck}, 'TARIFF_SWITCH', NULL, '{new_product}', '{status}', {ts}, " +
                    (f"{ts} + interval '2 days'" if status in ('CONFIRMED','ACTIVE') else "NULL") + f", CURRENT_DATE - {day_offset} + 30)")
            elif action == 'SERVICE_REQUEST':
                req_type = random.choice(request_types)
                subject = random.choice(request_subjects[req_type]).replace("'", "''")
                detail = f'{{"request_type": "{req_type}", "subject": "{subject}"}}'
                req_status = random.choices(['OPEN', 'CLOSED'], weights=[30, 70])[0]
                request_values.append(f"({ck}, '{req_type}', '{subject}', NULL, '{req_status}', 'NORMAL', {ts}, " +
                    (f"{ts} + interval '{random.randint(1,72)} hours'" if req_status == 'CLOSED' else "NULL") + ")")

            activity_values.append(f"({ck}, {ts}, '{action}', '{detail}', '{city}', '{region}', '{ctype}')")

    f.write('-- Portal activity log\n')
    for i in range(0, len(activity_values), 500):
        batch = activity_values[i:i+500]
        f.write(f"INSERT INTO portal_activity_log (customer_key, event_time, event_type, event_detail, city, region, customer_type) VALUES\n")
        f.write(',\n'.join(batch))
        f.write(';\n\n')

    f.write('-- Meter readings\n')
    for i in range(0, len(reading_values), 500):
        batch = reading_values[i:i+500]
        f.write(f"INSERT INTO meter_readings (customer_key, reading_date, meter_type, reading_kwh, submitted_at, source) VALUES\n")
        f.write(',\n'.join(batch))
        f.write(';\n\n')

    f.write('-- Tariff orders\n')
    for i in range(0, len(order_values), 500):
        batch = order_values[i:i+500]
        f.write(f"INSERT INTO tariff_orders (customer_key, order_type, current_product, requested_product, status, created_at, confirmed_at, effective_date) VALUES\n")
        f.write(',\n'.join(batch))
        f.write(';\n\n')

    f.write('-- Service requests\n')
    for i in range(0, len(request_values), 500):
        batch = request_values[i:i+500]
        f.write(f"INSERT INTO service_requests (customer_key, request_type, subject, description, status, priority, created_at, resolved_at) VALUES\n")
        f.write(',\n'.join(batch))
        f.write(';\n\n')

    f.write('COMMIT;\n')

print(f"\nGenerated {sql_path}")
print(f"  Activities:       {len(activity_values):,}")
print(f"  Meter readings:   {len(reading_values):,}")
print(f"  Tariff orders:    {len(order_values):,}")
print(f"  Service requests: {len(request_values):,}")
print(f"\nNext: run in psql:  psql service=my_epower_portal -f portal_seed_data.sql")

---

### ⏸️ Pause: Complete the Postgres setup before continuing

Run the following in your Postgres client (psql, DBeaver, pgAdmin):

1. **Schema + pg_lake setup:** `psql service=my_epower_portal -f portal_postgres_setup.sql`
2. **Seed data:** `psql service=my_epower_portal -f portal_seed_data.sql`
3. **Wait ~2 minutes** for pg_incremental to backfill the Iceberg table
4. **Verify** in psql:
   ```sql
   SELECT (SELECT count(*) FROM portal_activity_log) AS heap_rows,
          (SELECT count(*) FROM portal_activity_log_iceberg) AS iceberg_rows;
   ```

Once both counts match, continue with the next cell.

---

## 5. Snowflake Catalog Integration

Connect Snowflake to the Postgres-managed Iceberg table. After this, portal activity data is queryable in Snowflake — with auto-refresh every 30 seconds.

In [ ]:
%%sql
CREATE OR REPLACE CATALOG INTEGRATION PORTAL_POSTGRES_CATALOG
  CATALOG_SOURCE    = SNOWFLAKE_POSTGRES
  TABLE_FORMAT      = ICEBERG
  CATALOG_NAMESPACE = 'public'
  REST_CONFIG = (
    POSTGRES_INSTANCE      = 'MY_EPOWER_PORTAL'
    CATALOG_NAME           = 'postgres'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  ENABLED = TRUE;

In [ ]:
%%sql
CREATE OR REPLACE ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
    CATALOG = 'PORTAL_POSTGRES_CATALOG'
    CATALOG_TABLE_NAME = 'portal_activity_log_iceberg';

ALTER ICEBERG TABLE EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
    SET AUTO_REFRESH = TRUE;

In [ ]:
%%sql
SELECT
    count(*) AS total_events,
    count(DISTINCT customer_key) AS unique_customers,
    min(event_time) AS earliest,
    max(event_time) AS latest,
    count(CASE WHEN event_type = 'LOGIN' THEN 1 END) AS logins,
    count(CASE WHEN event_type = 'METER_READING' THEN 1 END) AS readings,
    count(CASE WHEN event_type = 'TARIFF_CHANGE' THEN 1 END) AS tariff_changes,
    count(CASE WHEN event_type = 'SERVICE_REQUEST' THEN 1 END) AS service_requests
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;

## 6. Analytics Model

We create a Gold-layer mart that aggregates portal activity into **daily engagement metrics** by region and customer type — answering questions like "How active is our portal?" and "Which regions have low digital adoption?"

In [ ]:
%%sql
CREATE OR REPLACE TABLE EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT AS
SELECT
    DATE_TRUNC('DAY', event_time)::DATE AS activity_date,
    region,
    customer_type,
    event_type,
    COUNT(*) AS event_count,
    COUNT(DISTINCT customer_key) AS unique_customers
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG
GROUP BY 1, 2, 3, 4
ORDER BY activity_date DESC, region, event_type;

In [ ]:
%%sql
SELECT activity_date, region, event_type, event_count, unique_customers
FROM EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT
ORDER BY activity_date DESC
LIMIT 20;

## 7. Semantic View + Agent Update

We create a **PORTAL_SEMANTIC_VIEW** and add a `portal_analyst` tool to the Intelligence Agent — enabling natural language questions about portal engagement.

In [ ]:
%%sql
CREATE OR REPLACE SEMANTIC VIEW EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW
  COMMENT = 'Customer portal engagement analytics — sourced from Snowflake Postgres via pg_lake'
AS
TABLES (
  ENGAGEMENT REFERENCES EPOWER_DEMO.EPOWER_GOLD.MART_PORTAL_ENGAGEMENT (
    PRIMARY KEY (ACTIVITY_DATE, REGION, CUSTOMER_TYPE, EVENT_TYPE)
    FACTS (
      EVENT_COUNT COMMENT 'Number of portal events'
        SYNONYMS ('events', 'activities', 'actions', 'Aktionen', 'Aktivitäten'),
      UNIQUE_CUSTOMERS COMMENT 'Distinct customers who performed this event type'
        SYNONYMS ('active users', 'aktive Nutzer', 'unique users', 'DAU')
    )
    DIMENSIONS (
      ACTIVITY_DATE COMMENT 'Date of portal activity'
        SYNONYMS ('date', 'day', 'Datum', 'Tag'),
      REGION COMMENT 'German geographic region (Nord, Süd, West, Ost)'
        SYNONYMS ('region', 'Gebiet', 'Bundesland'),
      CUSTOMER_TYPE COMMENT 'Customer segment (Privatkunde, Kleingewerbe, Gewerbekunde)'
        SYNONYMS ('segment', 'Kundentyp', 'customer segment'),
      EVENT_TYPE COMMENT 'Type of portal action: LOGIN, METER_READING, TARIFF_CHANGE, SERVICE_REQUEST'
        SYNONYMS ('action type', 'Aktionstyp', 'event', 'Ereignis')
    )
  )
);

In [ ]:
%%sql
CREATE OR REPLACE AGENT EPOWER_DEMO.EPOWER_GOLD.EPOWER_AGENT
WITH PROFILE='{ "display_name": "EPOWER AGENT" }'
FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    You are a data analyst for EPOWER Energie Deutschland.
    CRITICAL LANGUAGE RULE: You MUST always respond in the SAME language as the user's question.
    DATA ACCESS: Energy sales, billing/consumption, service tickets, HR data, day-ahead electricity market prices, VPP IoT telemetry, customer portal engagement, and documents.
  orchestration: |
    TOOL SELECTION:
    - Document questions → energy_docs_search, product_docs_search, service_docs_search
    - Consumption + products → customer_energy_analyst
    - Sales/contracts → energy_sales_analyst
    - Billing → billing_analyst
    - Service tickets → service_analyst
    - HR data → hr_analyst
    - Electricity market prices, day-ahead → epulse_prices_analyst
    - VPP telemetry, solar yield, battery SOC, grid import/export → vpp_telemetry_analyst
    - Portal activity, digital engagement, meter readings, tariff changes, logins → portal_analyst
tools:
  - tool_spec: {type: cortex_analyst_text_to_sql, name: energy_sales_analyst, description: "Contracts, products, sales, revenue"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: billing_analyst, description: "Consumption, billing, payments"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: customer_energy_analyst, description: "Consumption by product ownership"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: service_analyst, description: "Service tickets, complaints"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: hr_analyst, description: "HR data, salaries"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: market_prices_analyst, description: "Day-ahead electricity market prices"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: vpp_telemetry_analyst, description: "VPP IoT telemetry: solar yield, battery SOC, grid import/export"}
  - tool_spec: {type: cortex_analyst_text_to_sql, name: portal_analyst, description: "Customer portal engagement: logins, meter readings, tariff changes, service requests, VPP enrollments. Data from Snowflake Postgres via pg_lake."}
  - tool_spec: {type: cortex_search, name: energy_docs_search, description: "Energy policies, terms"}
  - tool_spec: {type: cortex_search, name: product_docs_search, description: "Product documentation"}
  - tool_spec: {type: cortex_search, name: service_docs_search, description: "Service handbook"}
  - tool_spec: {type: cortex_search, name: service_logs_search, description: "Historical tickets"}
  - tool_spec: {type: data_to_chart, name: data_to_chart, description: "Generate visualizations"}
tool_resources:
  energy_sales_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.ENERGY_SALES_SEMANTIC_VIEW"}
  billing_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.BILLING_SEMANTIC_VIEW"}
  customer_energy_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.CUSTOMER_ENERGY_SEMANTIC_VIEW"}
  service_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.SERVICE_SEMANTIC_VIEW"}
  hr_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.HR_SEMANTIC_VIEW"}
  market_prices_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.MARKET_PRICES_SEMANTIC_VIEW"}
  vpp_telemetry_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.EPULSE_VPP_SEMANTIC_VIEW"}
  portal_analyst: {semantic_view: "EPOWER_DEMO.EPOWER_GOLD.PORTAL_SEMANTIC_VIEW"}
  energy_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_ENERGY_DOCS", max_results: 5}
  product_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_PRODUCT_DOCS", max_results: 5}
  service_docs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_DOCS", max_results: 5}
  service_logs_search: {search_service: "EPOWER_DEMO.EPOWER_GOLD.SEARCH_SERVICE_LOGS", max_results: 5}
$$;

## 8. Verification & Demo

### Demo Questions for the Agent

| # | Question | What it tests |
|---|----------|---------------|
| 1 | *"How many customers used the portal this week?"* | Basic engagement metric |
| 2 | *"Welche Region hat die höchste Portal-Nutzung?"* | Regional comparison (German) |
| 3 | *"Show me the trend of meter reading submissions over the last 30 days"* | Time-series + charting |
| 4 | *"What are the most popular tariff switches?"* | Tariff change analysis |
| 5 | *"Compare portal engagement between residential and business customers"* | Segment comparison |
| 6 | *"Which customers submit meter readings but have never changed their tariff?"* | **Cross-tool**: portal + sales |

&nbsp;

> **Presenter tip:** Question 6 combines portal data (from Postgres) with sales data (from Snowflake-native tables) — demonstrating the unified platform value.

### Live Demo: Real-Time Sync

Simulate a customer using the portal RIGHT NOW, then watch the data appear in Snowflake within 30–60 seconds.

**In your Postgres client (psql / DBeaver), run:**

```sql
INSERT INTO portal_activity_log (customer_key, event_time, event_type, event_detail, city, region, customer_type)
SELECT
    customer_key,
    now(),
    (ARRAY['LOGIN', 'METER_READING', 'TARIFF_CHANGE', 'SERVICE_REQUEST'])[1 + (random() * 3)::int],
    '{"source": "live_demo"}',
    'Hamburg',
    'Nord',
    'Privatkunde'
FROM portal_users
ORDER BY random()
LIMIT 50;
```

Wait ~60 seconds, then run the next cell to verify the data arrived in Snowflake.

In [ ]:
%%sql
SELECT
    MAX(event_time) AS most_recent_event,
    DATEDIFF('second', MAX(event_time), CURRENT_TIMESTAMP()) AS seconds_ago,
    COUNT(*) AS total_rows
FROM EPOWER_DEMO.EPOWER_BRONZE.PORTAL_ACTIVITY_LOG;

---

## Summary

In this module you built a complete **web application backend → analytics** pipeline:

| What | How |
|------|-----|
| **Portal backend** | Snowflake Postgres — users, meter readings, tariff orders, service requests |
| **Zero-ETL replication** | pg_lake + pg_incremental → Iceberg (no middleware) |
| **Near real-time** | Auto-refresh every 30 seconds |
| **Analytics** | Gold-layer engagement metrics by region, segment, and action type |
| **AI-ready** | Semantic View + Cortex Agent — queryable in natural language |

**The key insight:** The portal's web application needs Postgres for what web apps always need — low-latency CRUD, session management, form validation, transactional consistency. Snowflake Postgres provides this as a managed service. The analytics-relevant activity stream flows to Snowflake automatically via open standards (Iceberg) — no ETL pipeline to build or maintain.

---

*EPOWER Module 2 — Snowflake Postgres + pg_lake — Powered by Snowflake*